# Domain Prediction

Multiclass

In [4]:
import pandas as pd
import numpy as np
import re

# Huggingface transformers
# import transformers
# from transformers import BertModel, BertConfig, BertTokenizer,AdamW, get_linear_schedule_with_warmup

from torch import nn ,cuda
from torch.utils.data import DataLoader,Dataset,RandomSampler, SequentialSampler

# import pytorch_lightning as pl
# from pytorch_lightning.callbacks import ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


# Handling html data
# from bs4 import BeautifulSoup

# import seaborn as sns
# from pylab import rcParams
import matplotlib.pyplot as plt
from matplotlib import rc
%matplotlib inline
import json
import ast


## Load data, prepare features

In [6]:
import os

# from google.colab import drive
# drive.mount('/content/drive',force_remount=True)

# root = 'drive/MyDrive/Legal NLP Project (with MPI Coll)/Colab_coding_files/latest_code'
root = '.'
# df_cleaned_final = pd.read_csv('data/df_cleaned_final_new.csv', index_col=[0])
df_cleaned_final = pd.read_csv(os.path.join(root,'court_case_data.csv'))
# df_cleaned_final = pd.read_csv(os.path.join(root,'bverfg.csv'))
# df_cleaned_final

df_cleaned_final = df_cleaned_final.drop_duplicates('full_text_cleaned').reset_index()
df = df_cleaned_final
df

,index,Unnamed: 0,uid,date_and_first_docket,date,year,ref_seidl,ref_papier,ref_haas,ref_grimm,...,dm_levies_judge_domain_score,dm_manifestations_judge_domain_score,dm_professions_judge_domain_score,dm_property_judge_domain_score,dm_socialsecurity_judge_domain_score,dm_speech_judge_domain_score,dm_tax_judge_domain_score,combined_domains_score,dummy_highest_prob,juid
0,0,0,1,08. Januar 1998 - 1 BvR 390/96,1998-01-08,1998,0,0,0,0,...,0.0,0.00,0.125,0.0,0.0,0.00,0.0,0.125,0,0
1,3,3,2,08. Januar 1998 - 1 BvR 1872/94,1998-01-08,1998,0,0,0,0,...,0.0,0.00,0.130,0.0,0.0,0.00,0.0,0.130,0,3
2,6,6,25,05. Februar 1998 - 1 BvR 410/95,1998-02-05,1998,0,0,0,1,...,0.0,0.06,0.000,0.0,0.0,0.01,0.0,0.200,0,6
3,9,9,26,05. Februar 1998 - 1 BvR 2666/95,1998-02-05,1998,0,0,0,1,...,0.0,0.08,0.000,0.0,0.0,0.04,0.0,0.220,0,9
4,12,12,29,06. Februar 1998 - 1 BvR 1788/97,1998-02-06,1998,0,0,0,0,...,0.0,0.00,0.135,0.0,0.0,0.00,0.0,0.135,0,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3183,10348,10348,8162,30. September 2022 - 2 BvR 2222/21,2022-09-30,2022,0,0,0,0,...,0.0,0.00,0.000,0.0,0.0,0.00,0.0,0.000,1,10348
3184,10351,10351,8164,30. September 2022 - 1 BvR 922/21,2022-09-30,2022,0,0,0,0,...,0.0,0.00,0.000,0.0,0.0,0.00,0.0,0.000,0,10351
3185,10354,10354,8183,04. November 2022 - 2 BvR 2202/19,2022-11-04,2022,0,0,0,0,...,0.0,0.00,0.000,0.0,0.0,0.00,0.0,0.000,0,10354
3186,10357,10357,8185,17. November 2022 - 2 BvR 2013/22,2022-11-17,2022,0,0,0,0,...,0.0,0.00,0.000,0.0,0.0,0.00,0.0,0.000,1,10357


In [23]:
list(df.columns)

['index',
 'Unnamed: 0',
 'uid',
 'date_and_first_docket',
 'date',
 'year',
 'ref_seidl',
 'ref_papier',
 'ref_haas',
 'ref_grimm',
 'ref_hoemig',
 'ref_kuehling',
 'ref_seibert',
 'ref_jaeger',
 'ref_steiner',
 'ref_hohmanndennhardt',
 'ref_hoffmannriem',
 'ref_bryde',
 'ref_gaier',
 'ref_schluckebier',
 'ref_eichberger',
 'ref_fkirchhof',
 'ref_masing',
 'ref_paulus',
 'ref_britz',
 'ref_baer',
 'ref_ott',
 'ref_christ',
 'ref_radtke',
 'ref_harbarth',
 'ref_haertel',
 'ref_wolff',
 'sumref',
 'ref2_kruis',
 'ref2_grasshof',
 'ref2_pkirchhof',
 'ref2_winter',
 'ref2_limbach',
 'ref2_sommer',
 'ref2_jentsch',
 'ref2_hassemer',
 'ref2_bross',
 'ref2_osterloh',
 'ref2_mellinghoff',
 'ref2_difabio',
 'ref2_luebbewolff',
 'ref2_gerhardt',
 'ref2_landau',
 'ref2_vosskuhle',
 'ref2_hermanns',
 'ref2_huber',
 'ref2_pmueller',
 'ref2_kessalwulf',
 'ref2_koenig',
 'ref2_maidowski',
 'ref2_langenfeld',
 'ref2_wallrabenstein',
 'sumref2',
 'dm_environmental_prob',
 'dm_tax_prob',
 'dm_property_

In [8]:
# case embeddings map

# torch.save(document_embeddings, os.path.join(root, 'data/case_embeddings_avg_map.pt'))
document_embeddings_map = torch.load(os.path.join(root, 'data/case_embeddings_avg_map.pt'))
len(document_embeddings_map)


# df_cleaned_final['case_embedding']
document_embeddings = df_cleaned_final['full_text_cleaned'].apply(lambda doc: document_embeddings_map[doc])
document_embeddings_avg = document_embeddings


document_embeddings_avg_numpy = [tensor.detach().cpu().numpy() for tensor in document_embeddings_avg]

numeric_array = np.array(df_cleaned_final[['prob','combined_domains_score']])

In [9]:
# judge candidate and judge domain features

import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import LabelEncoder

# def prepare_features(df):

#   df['judges'] = df.clean_judges.apply(lambda l: dict(zip(eval(l), [1,1,1])))

#   v = DictVectorizer(sparse=False)
#   # D = [{'foo': 1, 'bar': 2}, {'foo': 3, 'baz': 1}]
#   D = df.judges
#   encoded_judges = v.fit_transform(D)

#   le = LabelEncoder()
#   le.fit(df.ref_name_final)
#   # le.classes_
#   df_cleaned_final['target'] = le.transform(df.ref_name_final)
#   # le.inverse_transform([0, 0, 1, 2])

#   return df, encoded_judges

# df, encoded_judges = prepare_features(df_cleaned_final)


df['judges'] = df.clean_judges.apply(lambda l: dict(zip(eval(l), [1,1,1])))

dv = DictVectorizer(sparse=False)
D = df.judges
encoded_judges = dv.fit_transform(D)

le = LabelEncoder()
le.fit(df.ref_name_final)
# le.classes_
df_cleaned_final['target'] = le.transform(df.ref_name_final)
# le.inverse_transform([0, 0, 1, 2])


# encode judge domains


In [10]:
# domain encoder

def get_labels(d):
  res = {}
  for k,domains in d.items():
    for d in domains:
      res[f'{k}_{d}'] = 1
  return res

judge_domains = df.domains_of_judges.apply(lambda d: get_labels(eval(d)))

domain_vec = DictVectorizer(sparse=False)
encoded_domains = domain_vec.fit_transform(judge_domains)
encoded_domains.shape

(3188, 288)

In [11]:

# # Split the dictionary column into multiple columns
# # judges * domains
# pd.json_normalize(df.domains_of_judges.apply(eval))


In [12]:
# dv.get_feature_names_out()

# document_embeddings_avg_numpy_updated = np.concatenate([encoded_judges, numeric_array, document_embeddings_avg_numpy],axis=1)
doc_embedding_features = np.concatenate([encoded_judges, document_embeddings_avg_numpy],axis=1)

In [13]:
# TFIDF features

import scipy

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
cases_tfidf = vectorizer.fit_transform(df_cleaned_final['full_text_cleaned'])
# vectorizer.get_feature_names_out()
print(cases_tfidf.shape)


from scipy.sparse import hstack
# tfidf_features = hstack([scipy.sparse.csr_matrix(encoded_judges), cases_tfidf, scipy.sparse.csr_matrix(numeric_array)])
tfidf_features = hstack([scipy.sparse.csr_matrix(encoded_judges), cases_tfidf])


(3188, 118826)


In [14]:
# topic model features

selected_features_all = (df.columns.str.contains('prob'))|(df.columns=='combined_domains_score')
domain_features = (df.columns.str.contains('prob'))&(df.columns.str.contains('dm'))
# selected_features_all = (train_dataset.columns=='prob')| train_dataset.columns.str.contains('one_case_one_judge_domain_score')|\
#                         (train_dataset.columns=='combined_domains_score')
# judge_domain_score
# selected_features_no_prob = train_dataset.columns.str.contains('one_case_one_judge_domain_score')|\
#                         (train_dataset.columns=='combined_domains_score')
# selected_features_no_combined_domains_score = (train_dataset.columns=='prob')|train_dataset.columns.str.contains('one_case_one_judge_domain_score')
# selected_features_no_one_case_one_judge_domain_score = (train_dataset.columns=='prob')| (train_dataset.columns=='combined_domains_score')


tm_features = df.iloc[:,selected_features_all].to_numpy()
tm_features = np.concatenate((tm_features, encoded_judges), axis=1)
tm_features.shape

# selected_features_all
topic_model_feature_names = df.iloc[:,selected_features_all].columns
topic_model_feature_names

Index(['dm_environmental_prob', 'dm_tax_prob', 'dm_property_prob',
       'dm_labour_prob', 'dm2_crim_prob', 'dm2_publicservice_prob',
       'dm_family_prob', 'dm_corporations_prob', 'dm_socialsecurity_prob',
       'dm2_foreigner_prob', 'dm_manifestations_prob', 'dm_levies_prob',
       'dm2_municipalities_prob', 'dm_professions_prob',
       'dm2_extradition_prob', 'dm2_parliament_prob', 'dm2_prosecution_prob',
       'dm_freedomgeneral_prob', 'dm_speech_prob', 'dm2_detention_prob',
       'dm2_pretrial_prob', 'baer_prob', 'britz_prob', 'bross_prob',
       'brossss_prob', 'bryde_prob', 'christ_prob', 'difabio_prob',
       'eichberger_prob', 'fkirchhof_prob', 'gaier_prob', 'gerhard_prob',
       'gerhardt_prob', 'grasshof_prob', 'grimm_prob', 'haas_prob',
       'haertel_prob', 'harbarth_prob', 'hassemer_prob', 'hermanns_prob',
       'hoemig_prob', 'hoffmannriem_prob', 'hohmanndennhardt_prob',
       'huber_prob', 'jaeger_prob', 'jentsch_prob', 'kessalwulf_prob',
       'kessalwul

In [15]:
domains = [c.replace('_prob','') for c in df.iloc[:,domain_features].columns]
domains

['dm_environmental',
 'dm_tax',
 'dm_property',
 'dm_labour',
 'dm2_crim',
 'dm2_publicservice',
 'dm_family',
 'dm_corporations',
 'dm_socialsecurity',
 'dm2_foreigner',
 'dm_manifestations',
 'dm_levies',
 'dm2_municipalities',
 'dm_professions',
 'dm2_extradition',
 'dm2_parliament',
 'dm2_prosecution',
 'dm_freedomgeneral',
 'dm_speech',
 'dm2_detention',
 'dm2_pretrial']

In [16]:

# get train-test split
# If we split temporally then we need to handle unseen/rare judges in the test set

# train_size = 0.8
# train_dataset = df_cleaned_final.sample(frac=train_size,random_state=200)
# test_dataset = df_cleaned_final.drop(train_dataset.index).reset_index(drop=True)
# train_dataset = train_dataset.reset_index(drop=True)

# train_dataset = df_cleaned_final[df_cleaned_final['year']<=2017]
train_dataset = df_cleaned_final[df_cleaned_final['year']<=2020]
test_dataset = df_cleaned_final.drop(train_dataset.index)
# train_dataset = train_dataset.reset_index(drop=True)

print("FULL Dataset: {}".format(df_cleaned_final.shape))
print("TRAIN Dataset: {}".format(train_dataset.shape))
print("TEST Dataset: {}".format(test_dataset.shape))


FULL Dataset: (3188, 169)
TRAIN Dataset: (3010, 169)
TEST Dataset: (178, 169)


In [17]:
# check there is no train-test overlap
set(test_dataset.full_text_cleaned) & set(train_dataset.full_text_cleaned)

set()

In [18]:
# Drop test examples with judges that were rarely seen in train

# unseen_judges = set(test_dataset.ref_name_final) - set(train_dataset.ref_name_final)
keep_test_judges = []
for judge in set(test_dataset.ref_name_final):
  train_cnt = (train_dataset.ref_name_final == judge).sum()
  if train_cnt > 2:
    keep_test_judges.append(judge)
  print(judge, train_cnt)


langenfeld 27
ott 10
britz 82
kessalwulf 93
baer 135
pmueller 46
huber 60
harbarth 18
hermanns 37
koenig 45
radtke 25
wallrabenstein 3
paulus 74
christ 10
maidowski 89
haertel 8


In [19]:
test_dataset = test_dataset[test_dataset.ref_name_final.isin(keep_test_judges)]
test_dataset.shape

(178, 169)

In [20]:

train_index = train_dataset.index
test_index = test_dataset.index

In [21]:
%%time

# run train-test prediction experiments
# random guessing gets?

from sklearn.linear_model import LogisticRegression
import sklearn.metrics as mx
from sklearn.metrics import classification_report, accuracy_score

import numpy as np
from sklearn.dummy import DummyClassifier
from joblib import dump, load


def run_prediction_experiment(X, y, save_name):

    # X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    #   *[X, y], test_size=0.2, train_size=None, random_state=42, shuffle=True, stratify=None)

    # custom temporal split, to prevent lookahead bias
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # model = LogisticRegression(fit_intercept=True,max_iter = 100000)
    model = LogisticRegression()
    # model = LogisticRegression(solver = 'sag', random_state = 1)

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    # y_pred_prob = logmodel.predict_proba(X_test)[:,1]
    print(classification_report(y_test,y_pred))

    dump(model, os.path.join(root, f'{save_name}.joblib'))
    # # load(model, 'tfidf_logreg.joblib')
    return model

def run_train_test_experiment(X, y, train_index, test_index, save_name):

    # X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    #   *[X, y], test_size=0.2, train_size=None, random_state=42, shuffle=True, stratify=None)

    # custom temporal split, to prevent lookahead bias
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = LogisticRegression(fit_intercept=True,max_iter = 100000)
    # model = LogisticRegression()
    # model = LogisticRegression(solver = 'sag', random_state = 1)

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test,y_pred)
    print(classification_report(y_test,y_pred))

    dump(model, os.path.join(root, f'{save_name}.joblib'))
    # # load(model, 'tfidf_logreg.joblib')
    return acc


def run_train_test_experiment_trials(X, y, save_name):

    # custom temporal split, to prevent lookahead bias
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    accuracy_scores = []

    for random_state in [1, 2, 3]:
      # model = LogisticRegression(fit_intercept=True,max_iter = 100000)
      model = LogisticRegression(solver = 'sag', random_state = random_state)
      model.fit(X_train,y_train)
      y_pred = model.predict(X_test)
      accuracy = accuracy_score(y_test,y_pred)
      accuracy_scores.append(accuracy)
      print(random_state, accuracy)

    print(accuracy_scores, sum(accuracy_scores)/len(accuracy_scores))
    dump(model, os.path.join(root, f'{save_name}.joblib'))
    return model

CPU times: user 28.9 ms, sys: 0 ns, total: 28.9 ms
Wall time: 36.6 ms


In [22]:
# time series cross val

def get_train_test_split(df, year):

  train_dataset = df[df['year']<=year]
  # test_dataset = df_cleaned_final.drop(train_dataset.index)
  test_dataset = df[((df['year']>year) & (df['year']<year+3))]

  # Drop test examples with judges that are rarely seen in train
  keep_test_judges = []
  for judge in set(test_dataset.ref_name_final):
    train_cnt = (train_dataset.ref_name_final == judge).sum()
    if train_cnt > 2:
      keep_test_judges.append(judge)
      print(judge, train_cnt)
  test_dataset = test_dataset[test_dataset.ref_name_final.isin(keep_test_judges)]
  print(len(test_dataset))

  train_index = train_dataset.index
  test_index = test_dataset.index

  return train_index, test_index


def get_time_series_cv_results():
  return

acc_scores = []
for year in [2005, 2010, 2017, 2020]:

  # train_dataset = df[df['year']<=year]
  # # test_dataset = df_cleaned_final.drop(train_dataset.index)
  # test_dataset = df[((df['year']>year) & (df['year']<year+3))]

  # # Drop test examples with judges that are rarely seen in train
  # keep_test_judges = []
  # for judge in set(test_dataset.ref_name_final):
  #   train_cnt = (train_dataset.ref_name_final == judge).sum()
  #   if train_cnt > 2:
  #     keep_test_judges.append(judge)
  #     print(judge, train_cnt)
  # test_dataset = test_dataset[test_dataset.ref_name_final.isin(keep_test_judges)]
  # print(len(test_dataset))

  # train_index = train_dataset.index
  # test_index = test_dataset.index
  train_index, test_index = get_train_test_split(df, year)

  # run_train_test_experiment(X, y, train_index, test_index, save_name)

  # acc = run_train_test_experiment(
  #   tfidf_features,
  #   df.ref_name_final,
  #   train_index,
  #   test_index,
  #   "tfidf"
  # )
  # acc = run_train_test_experiment(
  #   tm_features,
  #   df.ref_name_final,
  #   train_index,
  #   test_index,
  #   "topic_model"
  # )
  # .74, .6, .72, .62
  acc = run_train_test_experiment(
    doc_embedding_features,
    df.ref_name_final,
    train_index,
    test_index,
    "doc_embeddings_avg"
  )
  # acc = run_train_test_experiment(
  #   np.concatenate([tm_features, doc_embedding_features,encoded_domains], axis=1),
  #   df.ref_name_final,
  #   train_index,
  #   test_index,
  #   "embedding+topic_model+domains"
  # )
  acc_scores.append(acc)

print(acc_scores)
sum(acc_scores)/len(acc_scores)

hoemig 34
mellinghoff 19
hohmanndennhardt 55
steiner 127
haas 31
difabio 23
bross 48
luebbewolff 15
hoffmannriem 99
bryde 43
gaier 22
hassemer 164
papier 59
230


/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result

                  precision    recall  f1-score   support

           bross       0.40      1.00      0.57        16
           bryde       1.00      0.91      0.95        11
         difabio       1.00      0.18      0.30        17
           gaier       0.71      0.71      0.71        17
            haas       1.00      1.00      1.00         1
        hassemer       0.79      1.00      0.88        44
          hoemig       1.00      1.00      1.00         1
    hoffmannriem       0.92      0.92      0.92        49
hohmanndennhardt       0.95      0.78      0.86        23
         jentsch       0.00      0.00      0.00         0
     luebbewolff       0.00      0.00      0.00         8
     mellinghoff       1.00      0.06      0.11        17
          papier       0.40      0.67      0.50         6
         steiner       0.86      0.90      0.88        20

        accuracy                           0.75       230
       macro avg       0.72      0.65      0.62       230
    weighted

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result

                  precision    recall  f1-score   support

           bross       0.00      0.00      0.00         0
         difabio       0.67      0.33      0.44         6
      eichberger       0.67      0.59      0.62        17
       fkirchhof       0.44      0.73      0.55        11
           gaier       0.46      0.86      0.60        14
        gerhardt       0.46      0.75      0.57         8
hohmanndennhardt       0.00      0.00      0.00         0
          landau       0.67      0.36      0.47        11
     luebbewolff       0.50      0.40      0.44         5
          masing       0.78      0.93      0.85        15
     mellinghoff       0.54      1.00      0.70         7
          papier       0.00      0.00      0.00         0
          paulus       0.00      0.00      0.00        12
    schluckebier       1.00      0.53      0.70        30
       vosskuhle       0.60      0.60      0.60        10

        accuracy                           0.60       146
       macro

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result

                precision    recall  f1-score   support

          baer       0.71      1.00      0.83        15
         britz       0.80      0.80      0.80         5
        christ       0.00      0.00      0.00         5
       haertel       0.44      0.57      0.50         7
      harbarth       0.83      0.83      0.83         6
      hermanns       0.70      0.88      0.78         8
         huber       0.23      0.50      0.32         6
    kessalwulf       0.50      0.87      0.63        15
        koenig       0.67      0.40      0.50        15
    langenfeld       0.86      0.75      0.80        16
     maidowski       0.43      0.60      0.50        10
           ott       1.00      0.33      0.50         6
        paulus       0.80      0.80      0.80        15
      pmueller       0.56      0.60      0.58        15
        radtke       0.82      0.88      0.85        16
     vosskuhle       0.00      0.00      0.00         0
wallrabenstein       0.00      0.00      0.00  

KeyboardInterrupt: 

In [ ]:
# !pip install deep_translator

# from deep_translator import GoogleTranslator

# # Use any translator you like, in this example GoogleTranslator
# translator = GoogleTranslator(source='de', target='en')

# translated = []
# for x in df_cleaned_final['full_text_cleaned']:
#   try:
#     translated.append(translator.translate(x))  # output -> Weiter so, du bist großartig
#   except Exception as e:
#     print(e)

# translated
# len(translated)

# import pickle

# with open(os.path.join(root, 'data/translated_cases.pkl'), 'rb') as f:
#   trans = pickle.load(f)

# trans = list(trans.items())
# len(trans)

In [ ]:
train_index, test_index = get_train_test_split(df, 2017)


masing 108
fkirchhof 143
vosskuhle 52
maidowski 52
kessalwulf 46
langenfeld 10
paulus 47
huber 42
koenig 15
pmueller 35
baer 88
hermanns 28
britz 66
eichberger 76
164


In [ ]:
run_prediction_experiment(
    encoded_judges,
    df.ref_name_final,
    "judges"
)

              precision    recall  f1-score   support

        baer       0.67      1.00      0.80        18
       britz       1.00      0.33      0.50         6
  eichberger       0.00      0.00      0.00         4
   fkirchhof       1.00      1.00      1.00        19
    hermanns       1.00      0.33      0.50         3
       huber       0.12      0.17      0.14         6
  kessalwulf       0.53      0.81      0.64        26
      koenig       0.00      0.00      0.00        14
  langenfeld       0.73      0.92      0.81        12
   maidowski       0.78      0.95      0.86        19
      masing       0.66      1.00      0.79        19
      paulus       0.00      0.00      0.00        11
    pmueller       0.00      0.00      0.00         6
   vosskuhle       0.00      0.00      0.00         1

    accuracy                           0.67       164
   macro avg       0.46      0.46      0.43       164
weighted avg       0.55      0.67      0.59       164



/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

LogisticRegression()

In [ ]:
%%time

run_prediction_experiment(
    tm_features,
    df.ref_name_final,
    "ATM_features"
)

              precision    recall  f1-score   support

        baer       0.71      0.94      0.81        18
       britz       0.80      0.67      0.73         6
  eichberger       0.00      0.00      0.00         4
   fkirchhof       1.00      1.00      1.00        19
    hermanns       0.00      0.00      0.00         3
       huber       0.33      0.33      0.33         6
  kessalwulf       0.54      0.73      0.62        26
      koenig       0.50      0.07      0.12        14
  langenfeld       0.73      0.92      0.81        12
   maidowski       0.82      0.95      0.88        19
      masing       0.66      1.00      0.79        19
      paulus       0.00      0.00      0.00        11
    pmueller       0.50      0.50      0.50         6
   vosskuhle       1.00      1.00      1.00         1

    accuracy                           0.70       164
   macro avg       0.54      0.58      0.54       164
weighted avg       0.61      0.70      0.63       164

CPU times: user 922 ms, s

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

LogisticRegression()

In [ ]:
%%time

# embeddings alone

run_prediction_experiment(
    doc_embedding_features,
    df.ref_name_final,
    "doc_embeddings_avg"
)

              precision    recall  f1-score   support

        baer       0.89      0.94      0.92        18
       britz       0.67      0.33      0.44         6
  eichberger       0.57      1.00      0.73         4
   fkirchhof       1.00      1.00      1.00        19
    hermanns       0.50      0.33      0.40         3
       huber       0.27      0.50      0.35         6
  kessalwulf       0.56      0.54      0.55        26
      koenig       0.64      0.50      0.56        14
  langenfeld       0.70      0.58      0.64        12
   maidowski       0.86      0.95      0.90        19
      masing       0.78      0.95      0.86        19
      paulus       0.83      0.45      0.59        11
    pmueller       0.40      0.33      0.36         6
   vosskuhle       0.50      1.00      0.67         1

    accuracy                           0.72       164
   macro avg       0.66      0.67      0.64       164
weighted avg       0.73      0.72      0.71       164

CPU times: user 4.77 s, s

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
%%time

run_prediction_experiment(
    np.concatenate([tm_features, doc_embedding_features], axis=1),
    df.ref_name_final,
    "embedding+topic_model"
)

# run_train_test_experiment_trials(
#     np.concatenate([tm_features, doc_embedding_features], axis=1),
#     df.ref_name_final,
#     "embedding+topic_model"
# )

              precision    recall  f1-score   support

        baer       0.94      0.94      0.94        18
       britz       0.71      0.83      0.77         6
  eichberger       0.75      0.75      0.75         4
   fkirchhof       1.00      1.00      1.00        19
    hermanns       1.00      0.33      0.50         3
       huber       0.22      0.33      0.27         6
  kessalwulf       0.61      0.65      0.63        26
      koenig       0.70      0.50      0.58        14
  langenfeld       0.73      0.92      0.81        12
   maidowski       0.82      0.95      0.88        19
      masing       0.73      1.00      0.84        19
      paulus       1.00      0.27      0.43        11
    pmueller       1.00      0.17      0.29         6
   vosskuhle       1.00      1.00      1.00         1

    accuracy                           0.76       164
   macro avg       0.80      0.69      0.69       164
weighted avg       0.79      0.76      0.74       164

CPU times: user 5.94 s, s

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
%%time

run_prediction_experiment(
    tfidf_features,
    df['target'],
    "tfidf"
)

              precision    recall  f1-score   support

           0       0.75      1.00      0.86        18
           1       1.00      0.67      0.80         6
           6       1.00      0.25      0.40         4
           7       1.00      1.00      1.00        19
          16       1.00      0.33      0.50         3
          20       0.00      0.00      0.00         6
          23       0.60      0.81      0.69        26
          24       1.00      0.36      0.53        14
          28       0.69      0.75      0.72        12
          31       0.78      0.95      0.86        19
          32       0.66      1.00      0.79        19
          37       0.00      0.00      0.00        11
          39       0.38      0.50      0.43         6
          45       0.00      0.00      0.00         1

    accuracy                           0.72       164
   macro avg       0.63      0.54      0.54       164
weighted avg       0.69      0.72      0.67       164

CPU times: user 3min 14s,

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

LogisticRegression()

In [ ]:
%%time

run_prediction_experiment(
    hstack([scipy.sparse.csr_matrix(encoded_judges), cases_tfidf, scipy.sparse.csr_matrix(tm_features)]),

    df['target'],
    "tfidf_tm"
)

              precision    recall  f1-score   support

           0       0.71      0.94      0.81        18
           1       1.00      0.67      0.80         6
           6       0.00      0.00      0.00         4
           7       1.00      1.00      1.00        19
          16       0.00      0.00      0.00         3
          20       0.50      0.33      0.40         6
          23       0.64      0.81      0.71        26
          24       1.00      0.43      0.60        14
          28       0.73      0.92      0.81        12
          31       0.82      0.95      0.88        19
          32       0.66      1.00      0.79        19
          37       0.00      0.00      0.00        11
          39       0.50      0.50      0.50         6
          45       1.00      1.00      1.00         1

    accuracy                           0.74       164
   macro avg       0.61      0.61      0.59       164
weighted avg       0.68      0.74      0.69       164

CPU times: user 3min 56s,

/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/amao/miniconda3/envs/nlp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

LogisticRegression()

In [ ]:
# model = load('ATM_features.joblib')
model = load(os.path.join(root, f'tfidf_tm.joblib'))
# dump(model, os.path.join(root, f'ATM_features.joblib'))

model

LogisticRegression()

In [ ]:
# feature importance

# Get the coefficients (weights) of the features
coefficients = [round(x,3) for x in model.coef_[0]]

# Get the names of the features
# hstack([scipy.sparse.csr_matrix(encoded_judges), cases_tfidf, scipy.sparse.csr_matrix(tm_features)]),
# feature_names = np.concatenate([dv.get_feature_names_out(), vectorizer.get_feature_names_out()])
feature_names = np.concatenate([dv.get_feature_names_out(), vectorizer.get_feature_names_out(), topic_model_feature_names])
assert len(feature_names) == len(coefficients)

# Create a dictionary mapping feature names to their corresponding coefficients
feature_coefficients = dict(zip(feature_names, coefficients))

# Print the feature names and their coefficients, sorted by coefficient magnitude
sorted_features = sorted(feature_coefficients.items(), key=lambda x: abs(x[1]), reverse=True)


AssertionError: 

In [ ]:
len(feature_names), len(coefficients)

: 

In [ ]:
sorted_features[:20]

: 

In [ ]:
# len(' '.join(feature_names))
top_feature_names = list(zip(*sorted_features))[0][:100]
# ' '.join(
# len(top_feature_names)

top_feature_names[:50]

: 

In [ ]:
# 'fkirchhof',
#  'bundesarbeitsgericht',
#  'landesarbeitsgerichts',
#  'bundesarbeitsgerichts',
#  'marburg',
#  'landesarbeitsgericht',
#  'gieen',
#  'beratungshilfe',
#  'universitt',
#  'der',
#  'sgb',
#  'as',
#  'tkg',
#  'ukg',
#  'sa',
#  'arbeitsgerichts',


# 'fkirchhof',
#  'federal labor court',
#  'state labor court',
#  'federal labor court',
#  'marburg',
#  'state labor court',
#  'pour',
#  'advisory assistance',
#  'university',
#  'the',
#  'sgb',
#  'as',
#  'tkg',
#  'ukg',
#  'sa',
#  'labor court',

: 

In [ ]:
# # !pip install deep_translator

# from deep_translator import GoogleTranslator

# # Use any translator you like, in this example GoogleTranslator
# translator = GoogleTranslator(source='de', target='en')

# translated = []
# for x in top_feature_names:
#   translated.append(translator.translate(x))  # output -> Weiter so, du bist großartig

# translated

: 

In [ ]:
# analyze failures

# list(df.columns)
df[df.ref_name_final == 'christ']

: 

In [ ]:
df

: 

In [ ]:
# # calculate domain scores for cases, using the ref and judge domains
# # model a case and domain scores

# # for a new case, predict domains, judge-domain score, referee
# # features: judges, judge domains, domain scores, "judge scores"

# for i,row in df.iterrows():
#   # row.ref_name_final

#   # row.ref_name_final

#   # print(eval(row.domains_of_judges))
#   print(eval(row.domains_of_judges)[row.ref_name_final])

#   break

: 

: 

In [ ]:
document_embeddings_avg_numpy_train = document_embeddings_avg_numpy[0:len(training_set_logistic['juid'])]
document_embeddings_avg_numpy_test = document_embeddings_avg_numpy[len(training_set_logistic['juid']):len(document_embeddings_avg_numpy)]
# true_label_train = df_cleaned_final[['ref_in_judge']][0:len(training_set_logistic['juid'])]
# true_label_test = df_cleaned_final[['ref_in_judge']][len(training_set_logistic['juid']):len(document_embeddings_avg_numpy)]
true_label_train = df_cleaned_final[['target']][0:len(training_set_logistic['juid'])]
true_label_test = df_cleaned_final[['target']][len(training_set_logistic['juid']):len(document_embeddings_avg_numpy)]


: 

In [ ]:
document_embeddings_avg_numpy_train_updated = document_embeddings_avg_numpy_updated[0:len(training_set_logistic['juid'])]
document_embeddings_avg_numpy_test_updated = document_embeddings_avg_numpy_updated[len(training_set_logistic['juid']):len(document_embeddings_avg_numpy_updated)]


: 

In [ ]:
training_set_logistic.clean_judges

: 

In [ ]:
training_set_logistic['embeddings'] = document_embeddings_avg_numpy_train

: 

In [ ]:
# Convert time to datetime
training_set_logistic['date'] = pd.to_datetime(training_set_logistic['date'])

# Group by author and calculate the average embedding before each time
training_embeddings_adjusted = training_set_logistic.groupby(['type', 'date']).apply(lambda x: x[x['date'] <= x['date'].max()]['embeddings'].sum()).reset_index()
training_embeddings_adjusted.columns = ['type','date','embeddings_adjusted']

: 

In [ ]:
training_set_logistic = pd.merge(training_set_logistic,training_embeddings_adjusted, how = 'left', on = ['type','date'])

: 

In [ ]:
testing_set_logistic['embeddings'] = document_embeddings_avg_numpy_test
# Convert time to datetime
testing_set_logistic['date'] = pd.to_datetime(testing_set_logistic['date'])

# Group by author and calculate the average embedding before each time
testing_embeddings_adjusted = testing_set_logistic.groupby(['type', 'date']).apply(lambda x: x[x['date'] <= x['date'].max()]['embeddings'].sum()).reset_index()
testing_embeddings_adjusted.columns = ['type','date','embeddings_adjusted']

: 

In [ ]:
testing_set_logistic = pd.merge(testing_set_logistic,testing_embeddings_adjusted, how = 'left', on = ['type','date'])

: 

In [ ]:
training_set_logistic

: 

In [ ]:
training_embeddings = np.array(training_set_logistic['embeddings_adjusted'].values.tolist())
testing_embeddings = np.array(testing_set_logistic['embeddings_adjusted'].values.tolist())

: 

In [ ]:
len(training_embeddings)

: 

In [ ]:
# from sklearn.linear_model import LogisticRegression
# logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
# logmodel.fit(document_embeddings_avg_numpy_train,true_label_train)
# y_pred = logmodel.predict(document_embeddings_avg_numpy_test)
# from sklearn.metrics import classification_report
# print(classification_report(true_label_test,y_pred))

: 

In [ ]:
# sum(y_pred)

: 

In [ ]:
# from sklearn.linear_model import LogisticRegression
# logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
# logmodel.fit(training_embeddings,true_label_train)
# y_pred = logmodel.predict(testing_embeddings)
# from sklearn.metrics import classification_report
# print(classification_report(true_label_test,y_pred))

: 

In [ ]:
sum(y_pred)

: 

In [ ]:
from sklearn.linear_model import LogisticRegression
logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
logmodel.fit(document_embeddings_avg_numpy_train_updated,true_label_train)
y_pred = logmodel.predict(document_embeddings_avg_numpy_test_updated)
y_pred_prob = logmodel.predict_proba(document_embeddings_avg_numpy_test_updated)[:,1]
from sklearn.metrics import classification_report
print(classification_report(true_label_test,y_pred))

: 

In [ ]:
def compute_case_accuracy(test_dataset,y_pred,y_pred_prob):
  test_metric = test_dataset[['uid','ref_in_judge']]
  test_metric['y_pred'] = y_pred
  test_metric['y_pred_prob'] = y_pred_prob
  test_metric['y_pred_nmax'] = (
      test_metric
      .groupby('uid')['y_pred_prob']
      .transform(
          lambda v: v.eq(v.max())
      )
      .astype(int)
  )
  print(test_metric)
  case_arr_naive = test_metric[test_metric['ref_in_judge']==1]['y_pred'].sum()/len(np.unique(test_metric.uid))
  case_arr = test_metric[test_metric['ref_in_judge']==1]['y_pred_nmax'].sum()/len(np.unique(test_metric.uid))
  print('case_arr_navie: ', case_arr_naive)
  print('case_arr: ', case_arr)
  return test_metric, case_arr_naive, case_arr

: 

In [ ]:
test_metric, case_arr_naive, case_arr = compute_case_accuracy(testing_set_logistic,y_pred,y_pred_prob)

: 

In [ ]:
training_embeddings_updated = np.concatenate([training_embeddings,numeric_array[0:len(training_set_logistic['juid'])]],axis=1)
testing_embeddings_updated = np.concatenate([testing_embeddings,numeric_array[len(training_set_logistic['juid']):]],axis=1)

: 

In [ ]:
from sklearn.linear_model import LogisticRegression
logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
logmodel.fit(training_embeddings_updated,true_label_train)
y_pred = logmodel.predict(testing_embeddings_updated)
y_pred_prob = logmodel.predict_proba(testing_embeddings_updated)[:,1]
from sklearn.metrics import classification_report
print(classification_report(true_label_test,y_pred))

: 

In [ ]:
test_metric, case_arr_naive, case_arr = compute_case_accuracy(testing_set_logistic,y_pred,y_pred_prob)

: 

In [ ]:
df_cleaned_final['embeddings'] = document_embeddings_avg_numpy
# Convert time to datetime
df_cleaned_final['date'] = pd.to_datetime(df_cleaned_final['date'])

# Group by author and calculate the average embedding before each time
df_embeddings_adjusted = df_cleaned_final.sort_values(['type', 'date']).groupby(['type', 'date']).apply(lambda x: x[x['date'] <= x['date'].max()]['embeddings'].sum()).reset_index()
df_embeddings_adjusted.columns = ['type','date','embeddings_adjusted']

df_cleaned_final_all = pd.merge(df_cleaned_final,df_embeddings_adjusted, how = 'left', on = ['type','date'])

: 

In [ ]:
df_embeddings_judge = np.array(df_cleaned_final_all['embeddings_adjusted'].values.tolist())

: 

In [ ]:
df_embeddings_judge

: 

In [ ]:
training_embedding_modify = df_embeddings_judge[0:len(training_set_logistic['juid'])]
testing_embedding_modify= df_embeddings_judge[len(training_set_logistic['juid']):len(document_embeddings_avg_numpy)]

: 

In [ ]:
from sklearn.linear_model import LogisticRegression
logmodel = LogisticRegression(fit_intercept=True,max_iter = 100000)
logmodel.fit(training_embedding_modify,true_label_train)
y_pred = logmodel.predict(testing_embedding_modify)
from sklearn.metrics import classification_report
print(classification_report(true_label_test,y_pred))

: 

## Binary Prediction With ATM Features

In [ ]:
# list of legal documents
legal_documents = []

# The labels for the samples.
labels = []

selected_features_domain_scores = df_cleaned_final.columns.str.contains('one_case_one_judge_domain_score')

for index, row in df_cleaned_final.iterrows():
    # Add the combined text to the list.
    combined_text = ''
    combined_text += '[SEP]'.join(row["clean_judges"])
    combined_text += "[SEP] {:} Author Probability [SEP] {:} ".format(row['clean_judges_final'],row["prob"])
    for i in range(df_cleaned_final.iloc[:,selected_features_domain_scores].shape[1]):
        column_name = df_cleaned_final.iloc[:,selected_features_domain_scores].columns[i]
        column_value = df_cleaned_final.iloc[index,selected_features_domain_scores][i]
        combined_text += "[SEP]"+ column_name+"[SEP]"+column_value.astype('str')
    combined_text += "[SEP] Combined Domain Scores [SEP] {:} ".format(row["combined_domains_score"])
    combined_text += row['full_text_cleaned']
    legal_documents.append(combined_text)

    # Also record the sample's label.
    labels.append(row["ref_in_judge"])

: 

In [ ]:
from transformers import BertTokenizer

# Load the BERT tokenizer.
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('dbmdz/bert-base-german-cased', do_lower_case=True)

: 

In [ ]:
from transformers import BertForSequenceClassification

# Load BertForSequenceClassification, the pretrained dbmdz/bert-base-german-cased BERT model
model = BertForSequenceClassification.from_pretrained(
    "dbmdz/bert-base-german-cased", # Use the dbmdz/bert-base-german-cased BERT model, with an cased vocab.
    num_labels = 2, # The number of output labels--2 for binary classification.
)

# Tell pytorch to run this model on the GPU.
desc = model.cuda()

: 

In [ ]:
# Larger batch sizes tend to be better, and we can fit this in memory.
batch_size = 32

# set learning rate
learning_rate = 1e-5

# Number of training epochs.
epochs = 4

# max length
max_len = 200

: 

In [ ]:
# Tokenize all of the sentences and map the tokens to thier word IDs.
input_ids = []
attention_masks = []

for doc in legal_documents:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
                        doc,                      # Sentence to encode.
                        add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                        max_length = max_len,           # Pad & truncate all sentences.
                        truncation = True,
                        padding = 'max_length',
                        return_attention_mask = True,   # Construct attn. masks.
                        return_tensors = 'pt',     # Return pytorch tensors.
                )

    # Add the encoded sentence to the list.
    input_ids.append(encoded_dict['input_ids'])

    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels)

print('DONE.')

: 

In [ ]:
training_set_logistic = pd.read_csv('data/input/training_set_logistic.csv')
testing_set_logistic = pd.read_csv('data/input/testing_set_logistic.csv')

: 

In [ ]:
train_idx = training_set_logistic['juid'].values
test_idx = testing_set_logistic['juid'].values

: 

In [ ]:
from torch.utils.data import TensorDataset

# Split the samples, and create TensorDatasets for each split.
train_dataset = TensorDataset(input_ids[train_idx], attention_masks[train_idx], labels[train_idx])
test_dataset = TensorDataset(input_ids[test_idx], attention_masks[test_idx], labels[test_idx])

: 

In [ ]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

# Create the DataLoaders for our training and validation sets.
# We'll take training samples in random order.
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

: 

In [ ]:
from transformers import AdamW

# Note: AdamW is a class from the huggingface library (as opposed to pytorch)
# I believe the 'W' stands for 'Weight Decay fix"
optimizer = AdamW(model.parameters(),
                  lr = learning_rate,
                  eps = 1e-8
                )

: 

In [ ]:
from transformers import get_linear_schedule_with_warmup

# Total number of training steps is [number of batches] x [number of epochs].
# (Note that this is not the same as the number of training samples!)
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

: 

In [ ]:
import numpy as np

# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

: 

In [ ]:
import time
import datetime

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))

    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

: 

In [ ]:
import pandas as pd

# Display floats with two decimal places.
#pd.set_option('precision', 2)

# Create a DataFrame from our training statistics.
df_stats = pd.DataFrame(data=training_stats)

# Use the 'epoch' as the row index.
df_stats = df_stats.set_index('epoch')

# A hack to force the column headers to wrap (doesn't seem to work in Colab).
#df = df.style.set_table_styles([dict(selector="th",props=[('max-width', '70px')])])

# Display the table.
df_stats

: 

In [ ]:
import torch.nn.functional as F

: 

In [ ]:
# Create a DataLoader to batch our test samples for us. We'll use a sequential
# sampler this time--don't need this to be random!
prediction_sampler = SequentialSampler(test_dataset)
prediction_dataloader = DataLoader(test_dataset, sampler=prediction_sampler, batch_size=batch_size)

print('Predicting labels for {:,} test sentences...'.format(len(test_dataset)))

# Put model in evaluation mode
model.eval()

# Tracking variables
predictions , prob, true_labels = [],[], []

# Predict
for batch in prediction_dataloader:
  # Add batch to GPU
  batch = tuple(t.to(device) for t in batch)

  # Unpack the inputs from our dataloader
  b_input_ids, b_input_mask, b_labels = batch

  # Telling the model not to compute or store gradients, saving memory and
  # speeding up prediction
  with torch.no_grad():
      # Forward pass, calculate logit predictions.
      result = model(b_input_ids,
                     token_type_ids=None,
                     attention_mask=b_input_mask,
                     return_dict=True)

  logits = result.logits
  probabilities = F.softmax(logits, dim=-1)


  # Move logits and labels to CPU
  logits = logits.detach().cpu().numpy()
  probabilities = probabilities.detach().cpu().numpy()
  label_ids = b_labels.to('cpu').numpy()

  # Store predictions and true labels
  predictions.append(logits)
  prob.append(probabilities)
  true_labels.append(label_ids)

print('    DONE.')

: 

In [ ]:
# Combine the results across all batches.
flat_predictions = np.concatenate(predictions, axis=0)

# For each sample, pick the label (0 or 1) with the higher score.
flat_predictions = np.argmax(flat_predictions, axis=1).flatten()

# Combine the correct labels for each batch into a single list.
flat_true_labels = np.concatenate(true_labels, axis=0)

: 

In [ ]:
from sklearn.metrics import f1_score

# Calculate the F1
f1 = f1_score(flat_true_labels, flat_predictions)

print('F1 Score: %.3f' % f1)

: 

In [ ]:
print(classification_report(flat_true_labels, flat_predictions))

: 

In [ ]:
flat_predictions

: 

In [ ]:
prob_flat = np.concatenate(prob, axis=0)

: 

In [ ]:
test_metric, case_arr_naive, case_arr = compute_case_accuracy(testing_set_logistic,flat_predictions,prob_flat[:,1])

: 

In [ ]:
model.save_pretrained("model/binary")

: 